# Mistral RAG Agent - Normalizacion CSV (MEJORADO)

- Normalizar dataset CSV
- Contar columnas y registros
- Agente RAG con LangChain + Mistral
- APIs modernas (sin deprecations)

## API Key

Ve al icono 🔑 **Secrets** en la barra izquierda de Colab.
Crea un secreto con **Name:** `mistralcollab` y pega tu key de https://console.mistral.ai.
Activa el toggle para que el notebook acceda.

In [ ]:
# 1. Instalar dependencias (versiones modernas)
!pip install -q \
  langchain \
  langchain-community \
  langchain-core \
  langchain-mistralai \
  langchain-chroma \
  chromadb \
  sentence-transformers \
  langchain-experimental \
  pandas \
  chardet

In [ ]:
# 2. Cargar API Key
import os

try:
    from google.colab import userdata
    MISTRAL_API_KEY = userdata.get('mistralcollab')
    print("API Key cargada desde Secrets")
except ImportError:
    import getpass
    MISTRAL_API_KEY = getpass.getpass("Ingresa tu Mistral API Key: ")
    print("API Key ingresada manualmente")

os.environ["MISTRAL_API_KEY"] = MISTRAL_API_KEY

In [ ]:
# 3. Subir CSV + Normalizar
import pandas as pd
from google.colab import files
import io
import chardet

print("Sube tu archivo CSV:")
uploaded = files.upload()

if not uploaded:
    raise Exception("No se subio ningun archivo")

filename = list(uploaded.keys())[0]
raw = uploaded[filename]

detected = chardet.detect(raw)
enc = detected['encoding']
print(f"Encoding detectado: {enc} ({detected['confidence']:.0%})")

encodings = [enc, 'cp1252', 'latin-1', 'ISO-8859-1', 'utf-8']
df = None
for e in encodings:
    if not e:
        continue
    try:
        df = pd.read_csv(io.BytesIO(raw), encoding=e, low_memory=False)
        if e != enc:
            print(f"Usando {e} (fallback)")
        break
    except UnicodeDecodeError:
        continue

if df is None:
    raise Exception("No se pudo leer el CSV")

# Usar pd.DataFrame() explicitamente para reconstruir el DataFrame
datos = [list(row) for row in df.itertuples(index=False)]
df = pd.DataFrame(datos, columns=list(df.columns))
print(f"DataFrame creado explicitamente con pd.DataFrame() - {df.shape[0]} filas x {df.shape[1]} columnas")

print(f"\n{'='*50}")
print(f"Archivo: {filename}")
print(f"Registros: {df.shape[0]} | Columnas: {df.shape[1]}")
print(f"Columnas: {list(df.columns)}")
print(f"{'='*50}")
print("\nPrimeras 5 filas:")
display(df.head())

# NORMALIZACION
print("\n" + "="*50)
print("NORMALIZANDO...")
print("="*50)
print("\nValores nulos:")
print(df.isnull().sum())

df_clean = df.copy()
df_clean = df_clean.dropna(how='all')
df_clean = df_clean.dropna(axis=1, how='all')

dup = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()

for col in df_clean.columns:
    if df_clean[col].dtype == 'object':
        df_clean[col] = df_clean[col].fillna('')
    else:
        df_clean[col] = df_clean[col].fillna(0)

df_clean.columns = (
    df_clean.columns.str.strip().str.lower()
    .str.replace(r'\s+', '_', regex=True)
    .str.replace(r'[^a-z0-9_]', '', regex=True)
)

print(f"\nDuplicados eliminados: {dup}")
print(f"Filas: {df_clean.shape[0]} | Columnas: {df_clean.shape[1]}")
print(f"\nColumnas normalizadas: {list(df_clean.columns)}")
display(df_clean.head())

print("Datatypes predominantes:")
print(df_clean.dtypes.value_counts())

In [ ]:
# 4. Elegir modelo gratuito de Mistral
MODELO_MISTRAL = "open-mistral-7b"
print(f"Modelo: {MODELO_MISTRAL}")
print(f"Alternativas gratuitas: open-mixtral-8x7b, open-mistral-7b, mistral-small-latest")

In [ ]:
# 5. RAG + Pandas Agent hibrido (Tool Calling + busqueda semantica)
from langchain_mistralai import ChatMistralAI
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.tools import tool
from langchain_experimental.agents import create_pandas_dataframe_agent

# --- INDEXAR EN VECTOR STORE (RAG) ---
docs = []
for idx, row in df_clean.iterrows():
    t = "\n".join([f"{c}: {row[c]}" for c in df_clean.columns])
    docs.append(Document(page_content=t, metadata={"fila": int(idx)}))
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(docs)
print(f"Documentos: {len(docs)} -> Chunks: {len(chunks)}")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_csv_db"
)
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)
print("Vector store listo")

# --- CONFIGURAR LLM ---
llm = ChatMistralAI(model=MODELO_MISTRAL, temperature=0.3, max_tokens=2048)
print(f"LLM: {MODELO_MISTRAL}")

# --- HERRAMIENTA RAG (busqueda semantica) ---
@tool
def consultar_csv_rag(question: str) -> str:
    """Obtiene contexto relevante del CSV mediante busqueda semantica. Usa esta herramienta cuando la pregunta requiera entender el significado o encontrar informacion especifica en los datos."""
    docs = retriever.invoke(question)
    return "\n\n".join([d.page_content for d in docs])

# --- AGENTE HIBRIDO (Tool Calling + RAG como extra_tool) ---
agent = create_pandas_dataframe_agent(
    llm=llm,
    df=df_clean,
    agent_type="tool-calling",
    allow_dangerous_code=True,
    verbose=True,
    include_df_in_prompt=True,
    number_of_head_rows=5,
    extra_tools=[consultar_csv_rag],
    prefix=(
        "Eres un asistente experto en analisis de datos CSV.\n"
        "Tienes acceso a DOS herramientas:\n"
        "1. Python REPL: ejecuta codigo pandas directamente sobre el DataFrame para calculos exactos.\n"
        "2. consultar_csv_rag: busca por significado en el vector store para entender el contenido.\n"
        "Usa Python REPL para preguntas de calculo (sumas, promedios, maximo, minimo, filtros).\n"
        "Usa consultar_csv_rag para preguntas de comprension o cuando no sepas que columna usar.\n"
        "NO inventes datos. Si no hay suficiente informacion, indicalo claramente."
    )
)

print("\n" + "="*50)
print("AGENTE HIBRIDO (RAG + TOOL CALLING) LISTO")
print("="*50)

In [ ]:
# 6. Hacer preguntas con Tool Calling
pregunta = "¿Qué información contiene el CSV? Dame un resumen general."

respuesta = agent.invoke({"input": pregunta})

print("\n" + "="*50)
print("RESPUESTA")
print("="*50)
print(respuesta["output"])

In [ ]:
# 7. Otra pregunta de ejemplo
pregunta2 = "¿Cuál es el producto más caro (mayor precio) en los datos?"

respuesta2 = agent.invoke({"input": pregunta2})

print("\n" + "="*50)
print("RESPUESTA")
print("="*50)
print(respuesta2["output"])

In [ ]:
# 8. Descargar CSV normalizado
df_clean.to_csv("dataset_normalizado.csv", index=False)
files.download("dataset_normalizado.csv")
print("Descargando...")